# Modul 5 · Zufall simulieren: Wahrscheinlichkeit, Bootstrap, Permutation
**Data Science 1** · linkit courses

Ist L-05 unzuverlässiger als L-01, oder hatte der Lieferant mit nur 30 Lieferungen einfach Pech? Wie oft steht Linie 1 länger als vier Stunden still? Wie genau ist ein Median aus 30 Werten? In diesem Notebook lässt du den Zufall selbst Daten erzeugen und siehst, wie stark Kennzahlen schwanken. Am Ende beantwortest du die Frage aus Modul 4: echt oder Zufall?

**Lernziele**
1. Zufallsgrößen im Werk passenden Verteilungen zuordnen und Wahrscheinlichkeiten berechnen
2. Wahrscheinlichkeiten per Monte-Carlo-Simulation schätzen und ihre Genauigkeit angeben
3. das Gesetz der großen Zahlen und den zentralen Grenzwertsatz per Simulation erklären
4. Standardfehler berechnen und von der Standardabweichung unterscheiden
5. Konfidenzintervalle mit dem Bootstrap bestimmen und seine Grenzen kennen
6. einen Permutationstest durchführen und den p-Wert richtig deuten
7. Simulationen mit Seeds reproduzierbar machen

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt.
- Mindestziel sind TODO 1 bis 3, TODO 5 bis 7, TODO 9 sowie TODO 11 bis 13. TODO 4, TODO 8, TODO 10 und die Bonusaufgaben vertiefen.
- Jede TODO-Zelle mit Zufall legt ihren eigenen Generator mit festem Seed an, `rng = np.random.default_rng(...)`. So liefert sie bei jedem Ausführen dasselbe. Die Selbsttests lassen so viel Spielraum, dass jeder richtige Weg besteht, auch wenn du in anderer Reihenfolge ziehst.
- Jede Grafikzelle endet mit `plt.show()`. Die Selbsttests prüfen bei Grafiken die Achsen und die Daten dahinter, nicht das Aussehen.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["lieferungen.csv", "messwerte.csv", "stoerungen.csv", "produktion.csv", "maschinen.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Zufall im Werk: Binomial, Poisson, Exponential, Normal
*Lerneinheit 1 · Skript Abschnitt 1*

Vier Verteilungen beschreiben vier Größen im Nordwerk:

| Verteilung | $P(X = k)$ oder Dichte $f(x)$ | $E(X)$ | $\mathrm{Var}(X)$ | Größe im Werk |
|---|---|---|---|---|
| Binomial | $\binom{n}{k} p^k (1-p)^{n-k}$ | $np$ | $np(1-p)$ | Mängel unter $n$ geprüften Teilen |
| Poisson | $\frac{\lambda^k}{k!} e^{-\lambda}$ | $\lambda$ | $\lambda$ | Störungen je Maschine und Schicht |
| Exponential | $\lambda e^{-\lambda x}$ | $1/\lambda$ | $1/\lambda^2$ | Verzug über die 2-Stunden-Grenze hinaus |
| Normal | $\frac{1}{\sigma\sqrt{2\pi}} e^{-(x-\mu)^2/(2\sigma^2)}$ | $\mu$ | $\sigma^2$ | Wellendurchmesser |

Die Formeln rechnest du mit `math.comb`, `math.exp`, `math.factorial` und `math.erf`. SciPy läuft im Browser nicht mit, im Beruf nimmst du meist `scipy.stats`. Neu sind die Tabellen `lieferungen.csv` und `messwerte.csv`, beschrieben in `DATENBLATT.md`. Als verspätet gilt eine Lieferung mit `verzug_h > 2`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SCHICHT_TYP = pd.CategoricalDtype(["Früh", "Spät", "Nacht"], ordered=True)
lieferungen = pd.read_csv("daten/lieferungen.csv")
messwerte = pd.read_csv("daten/messwerte.csv", dtype={"schicht": SCHICHT_TYP})
stoerungen = pd.read_csv("daten/stoerungen.csv", dtype={"schicht": SCHICHT_TYP})
produktion = pd.read_csv("daten/produktion.csv", dtype={"schicht": SCHICHT_TYP})
maschinen = pd.read_csv("daten/maschinen.csv")
print(lieferungen.shape, messwerte.shape, stoerungen.shape, produktion.shape)
print(lieferungen.groupby("lieferant").size().to_dict())

### TODO 1 · Größen den Verteilungen zuordnen
Ordne jeder Größe die Verteilung aus `VERTEILUNGEN` zu, die sie am besten beschreibt. Schreib die Namen genau so wie in der Liste.

- **a** Zahl der mangelhaften Teile unter den 80 geprüften Teilen einer großen Lieferung von L-02
- **b** Zahl der Störungen an der Teilewaschanlage M-08 in einer Nachtschicht
- **c** Wartezeit bis zur nächsten Störung an M-04, wenn Störungen unabhängig mit fester Rate auftreten
- **d** Durchmesser einer zufällig gewählten Welle von M-09
- **e** Zahl der verspäteten unter den nächsten 20 Lieferungen von L-06
- **f** mittlerer Verzug aus 200 zufällig gewählten Lieferungen von L-01, näherungsweise

`verteilung` ist ein Dictionary mit den Schlüsseln `"a"` bis `"f"`. Der Selbsttest vergleicht verschlüsselt, damit er die Antworten nicht verrät.

In [ ]:
VERTEILUNGEN = ["Binomial", "Poisson", "Exponential", "Normal"]

# TODO 1
verteilung = {"a": None, "b": None, "c": None, "d": None, "e": None, "f": None}

print(verteilung)

In [ ]:
import hashlib

def fingerabdruck(antworten):
    """Kurzer Fingerabdruck einer Liste von Antworten, damit der Test sie nicht verrät."""
    return hashlib.sha256("|".join(antworten).encode("utf-8")).hexdigest()[:10]

check("TODO 1a alles aus der Liste", lambda: sorted(verteilung) == list("abcdef") and all(v in VERTEILUNGEN for v in verteilung.values()))
check("TODO 1b Zuordnung", lambda: fingerabdruck([verteilung[k] for k in "abcdef"]) == "ed0a1b1094")

Bei **e** zählst du Erfolge in einer festen Zahl von Versuchen wie bei **a**, das ist binomial. Bei **f** hilft der zentrale Grenzwertsatz aus Abschnitt 3: Ein Mittelwert aus vielen Werten ist näherungsweise normalverteilt, auch wenn die einzelnen Verzüge es nicht sind.

**Binomial.** Die Zahl der Mängel unter $n$ geprüften Teilen ist binomialverteilt, wenn die Teile unabhängig voneinander mangelhaft sind und alle dieselbe Mangelquote $p$ haben. $P(X \ge 2)$ rechnest du über das Gegenereignis: $1 - P(X = 0) - P(X = 1)$. Für L-03 wie im Skript:

In [ ]:
l03 = lieferungen[lieferungen["lieferant"] == "L-03"]
p_l03 = l03["mangel_stueck"].sum() / l03["geprueft_stueck"].sum()
p_ab2_l03 = 1 - (1 - p_l03) ** 50 - 50 * p_l03 * (1 - p_l03) ** 49
fuenfzig_l03 = l03[l03["geprueft_stueck"] == 50]
print(round(p_l03, 4), round(p_ab2_l03, 4), len(fuenfzig_l03), round((fuenfzig_l03["mangel_stueck"] >= 2).mean(), 4))

### TODO 2 · Mindestens zwei Mängel bei L-01
1. `p_l01`: die Mangelquote von L-01 aus allen geprüften Teilen, also alle Mängel geteilt durch alle geprüften Teile.
2. `p_formel_l01`: $P(X \ge 2)$ für eine Stichprobe von 50 Teilen nach der Binomialverteilung.
3. `x_binom`: 100.000 simulierte Stichproben mit `rng.binomial(50, p_l01, size=...)`, und `p_sim_l01`: der Anteil mit mindestens zwei Mängeln.
4. `p_beob_l01`: der beobachtete Anteil mit mindestens zwei Mängeln unter den Lieferungen von L-01, bei denen 50 Teile geprüft wurden.

In [ ]:
# TODO 2a und 2b
p_l01 = None
p_formel_l01 = None

# TODO 2c
rng = np.random.default_rng(2)
x_binom = None
p_sim_l01 = None

# TODO 2d
p_beob_l01 = None

print(p_l01, p_formel_l01, p_sim_l01, p_beob_l01)

In [ ]:
check("TODO 2a p_l01", lambda: math.isclose(p_l01, 0.010391, abs_tol=1e-6))
check("TODO 2b p_formel_l01", lambda: math.isclose(p_formel_l01, 0.09540, abs_tol=1e-5))
check("TODO 2c Simulation", lambda: len(x_binom) >= 10_000 and 0 <= x_binom.min() and x_binom.max() <= 50
      and abs(p_sim_l01 - 0.09540) <= 4 * math.sqrt(0.0954 * 0.9046 / len(x_binom)))
check("TODO 2d p_beob_l01", lambda: math.isclose(p_beob_l01, 0.10298, abs_tol=1e-5))

Formel und Simulation liegen bei 9,5 %, beobachtet sind 10,3 % der 738 Lieferungen. Die Simulation weicht von der Formel nur um ihren Monte-Carlo-Fehler ab, die Beobachtung zusätzlich um den Zufall in den Daten und um alles, was das Modell nicht abbildet.

**Poisson.** Bei der Poisson-Verteilung sind Erwartungswert und Varianz gleich, $E(X) = \mathrm{Var}(X) = \lambda$. Wer Störungen je Schicht zählt, muss die Schichten **ohne** Störung ergänzen, denn `stoerungen.csv` hat nur Zeilen für Störungen. Alle gearbeiteten Schichten stehen in `produktion.csv`. Für M-03 wie im Skript:

In [ ]:
st_m03 = stoerungen[stoerungen["maschine"] == "M-03"]
nur_vorhandene = st_m03.groupby(["datum", "schicht"], observed=True).size()
alle_schichten = pd.MultiIndex.from_frame(produktion[["datum", "schicht"]].drop_duplicates())
zaehl_m03 = nur_vorhandene.reindex(alle_schichten, fill_value=0)
print(len(nur_vorhandene), round(nur_vorhandene.mean(), 2))
print(len(zaehl_m03), round(zaehl_m03.mean(), 3), round(zaehl_m03.var(), 3))

lam_m03 = zaehl_m03.mean()
beobachtet = zaehl_m03.value_counts().sort_index()
erwartet = [len(zaehl_m03) * lam_m03 ** k * math.exp(-lam_m03) / math.factorial(k) for k in beobachtet.index]
print(pd.DataFrame({"beobachtet": beobachtet, "erwartet": np.round(erwartet, 1)}))

### TODO 3 · Störungen an M-09 mit allen Schichten
Die neue Drehmaschine M-09 steht in derselben Linie wie M-03.
1. `zaehl_m09`: die Zahl der Störungen an M-09 je Schicht, mit `alle_schichten` als Index und 0 für Schichten ohne Störung. Dazu `lam_m09` (Mittelwert) und `var_m09` (Varianz).
2. `tabelle_m09`: ein DataFrame mit dem Index 0 bis 4 und den Spalten `beobachtet` (Zahl der Schichten mit so vielen Störungen) und `erwartet` (Zahl der Schichten nach der Poisson-Verteilung mit $\lambda$ = `lam_m09`).
3. Zeichne in `fig_pois, ax_pois` die beobachteten Zahlen als Balken und die erwarteten als Punkte, mit beschrifteten Achsen.

In [ ]:
# TODO 3a
zaehl_m09 = None
lam_m09 = None
var_m09 = None

# TODO 3b
tabelle_m09 = None

# TODO 3c
fig_pois, ax_pois = plt.subplots(figsize=(8, 4.5))

plt.show()
print(lam_m09, var_m09)
print(tabelle_m09)

In [ ]:
check("TODO 3a zaehl_m09 mit Nullen", lambda: len(zaehl_m09) == 1447 and int(zaehl_m09.sum()) == 736
      and math.isclose(lam_m09, 0.50864, abs_tol=1e-5) and math.isclose(var_m09, 0.49906, abs_tol=1e-5))
check("TODO 3b tabelle_m09", lambda: list(tabelle_m09.index) == [0, 1, 2, 3, 4]
      and [int(v) for v in tabelle_m09["beobachtet"]] == [867, 446, 114, 18, 2]
      and math.isclose(tabelle_m09["erwartet"].iloc[0], 870.10, abs_tol=0.01) and math.isclose(tabelle_m09["erwartet"].iloc[2], 112.55, abs_tol=0.01))
check("TODO 3c Balken und Punkte", lambda: len(ax_pois.patches) >= 5 and len(ax_pois.lines) >= 1
      and ax_pois.get_xlabel() != "" and ax_pois.get_ylabel() != "")

M-09 hat mit 0,51 Störungen je Schicht deutlich weniger als M-03 mit 0,88, und Mittelwert und Varianz liegen wieder dicht beieinander. Ohne die 867 Schichten ohne Störung käme ein Mittelwert von 1,27 heraus.

**Exponential.** Der Verzug über die 2-Stunden-Grenze hinaus, $T = \text{verzug} - 2$ für verspätete Lieferungen, ist bei L-01 näherungsweise exponentialverteilt. Die Exponentialverteilung ist **gedächtnislos**: $P(T > s + t \mid T > s) = P(T > t)$. Der mittlere weitere Verzug hängt also nicht davon ab, wie lange eine Lieferung schon überfällig ist. Für L-01 wie im Skript:

In [ ]:
verzug_l01 = lieferungen.loc[lieferungen["lieferant"] == "L-01", "verzug_h"]
beta_l01 = (verzug_l01[verzug_l01 > 2] - 2).mean()
for grenze in [2, 12, 24]:
    rest = verzug_l01[verzug_l01 > grenze] - grenze
    print(grenze, len(rest), round(rest.mean(), 2))

### TODO 4 · Gedächtnislosigkeit bei L-03
1. `rest_l03`: der mittlere weitere Verzug des Elektroniklieferanten L-03 über 2, 12 und 24 Stunden hinaus, als Series mit den Grenzen 2, 12 und 24 als Index. `anzahl_rest_l03`: wie viele Lieferungen jeweils dazu beitragen, in derselben Form.
2. `lam_l03`: die Rate $\hat\lambda = 1/\text{mittlerer Verzug über 2 Stunden hinaus}$.
3. `p_ueber24_modell`: die Wahrscheinlichkeit nach der Exponentialverteilung, dass eine verspätete Lieferung mehr als 24 Stunden Verzug hat, also $P(T > 22) = e^{-22\hat\lambda}$. `p_ueber24_beob`: der beobachtete Anteil der verspäteten Lieferungen von L-03 mit `verzug_h > 24`.

In [ ]:
# TODO 4a
rest_l03 = None
anzahl_rest_l03 = None

# TODO 4b und 4c
lam_l03 = None
p_ueber24_modell = None
p_ueber24_beob = None

print(rest_l03)
print(p_ueber24_modell, p_ueber24_beob)

In [ ]:
check("TODO 4a rest_l03", lambda: list(rest_l03.index) == [2, 12, 24]
      and all(math.isclose(a, b, abs_tol=1e-3) for a, b in zip(rest_l03, [25.337, 25.666, 26.712]))
      and [int(v) for v in anzahl_rest_l03] == [345, 229, 139])
check("TODO 4b lam_l03", lambda: math.isclose(lam_l03, 1 / 25.3368, rel_tol=1e-4))
check("TODO 4c Modell und Beobachtung", lambda: math.isclose(p_ueber24_modell, 0.41966, abs_tol=1e-4)
      and math.isclose(p_ueber24_beob, 0.40290, abs_tol=1e-4))

Auch bei L-03 bleibt der mittlere Rest zwischen 25 und 27 Stunden, egal wie lange die Lieferung schon überfällig ist. L-03 ist im Mittel aber deutlich langsamer als L-01. Das Modell sagt für mehr als 24 Stunden Verzug 42,0 % der verspäteten Lieferungen voraus, beobachtet sind 40,3 %.

**Normal.** Mit der Standardisierung $z = (x - \mu)/\sigma$ und der Verteilungsfunktion $\Phi(z) = \tfrac{1}{2}\big(1 + \mathrm{erf}(z/\sqrt{2})\big)$ rechnest du Anteile einer Normalverteilung aus. Die Toleranz der Wellen reicht von 24,975 bis 25,025 mm. Für M-03 wie im Skript:

In [ ]:
def Phi(z):
    """Verteilungsfunktion der Standardnormalverteilung."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

m03 = messwerte.loc[messwerte["maschine"] == "M-03", "durchmesser_mm"]
mu_m03, sigma_m03 = m03.mean(), m03.std()
z_unten, z_oben = (24.975 - mu_m03) / sigma_m03, (25.025 - mu_m03) / sigma_m03
p_aussen_m03 = Phi(z_unten) + 1 - Phi(z_oben)
beob_aussen_m03 = ((m03 < 24.975) | (m03 > 25.025)).mean()
print(round(mu_m03, 4), round(sigma_m03, 4), round(z_unten, 2), round(z_oben, 2))
print(round(Phi(z_unten), 4), round(1 - Phi(z_oben), 4), round(p_aussen_m03, 4), round(beob_aussen_m03, 4))

### TODO 5 · Toleranz bei M-09 und eine zentrierte M-03
1. `p_aussen_m09`: der Anteil der Wellen von M-09 außerhalb der Toleranz nach der Normalverteilung, mit $\mu$ und $\sigma$ aus den Messungen von M-09. `beob_aussen_m09`: der beobachtete Anteil.
2. `p_zentriert_m03`: Wie groß wäre der Anteil außerhalb, wenn M-03 mit ihrer Standardabweichung `sigma_m03` genau auf 25,000 mm arbeiten würde?

Eine Welle mit genau 24,975 oder 25,025 mm liegt noch in der Toleranz.

In [ ]:
# TODO 5a
p_aussen_m09 = None
beob_aussen_m09 = None

# TODO 5b
p_zentriert_m03 = None

print(p_aussen_m09, beob_aussen_m09, p_zentriert_m03)

In [ ]:
check("TODO 5a M-09", lambda: math.isclose(p_aussen_m09, 0.001467, abs_tol=2e-6) and math.isclose(beob_aussen_m09, 5 / 2826, abs_tol=1e-9))
check("TODO 5b zentrierte M-03", lambda: math.isclose(p_zentriert_m03, 0.04036, abs_tol=1e-5))

Zentriert hätte M-03 rund 4,0 % statt 5,0 % Wellen außerhalb der Toleranz. Den größeren Teil des Unterschieds zu M-09 macht die Streuung: Mit $\sigma = 0{,}0122$ mm liegt die Toleranzgrenze nur gut zwei Standardabweichungen vom Sollmaß entfernt, bei M-09 mehr als drei.

**Stichproben mit NumPy.** Ein Generator `rng = np.random.default_rng(seed)` zieht mit `rng.binomial(n, p, size)`, `rng.poisson(lam, size)`, `rng.exponential(scale, size)` und `rng.normal(loc, scale, size)`. Achtung: `exponential` erwartet den Mittelwert $1/\lambda$, `normal` die Standardabweichung.

In [ ]:
rng = np.random.default_rng(2026)
ziehungen = pd.DataFrame({
    "binomial": rng.binomial(50, p_l03, size=100_000),
    "poisson": rng.poisson(lam_m03, size=100_000),
    "exponential": rng.exponential(beta_l01, size=100_000),
    "normal": rng.normal(mu_m03, sigma_m03, size=100_000),
})
erwartung = pd.Series({"binomial": 50 * p_l03, "poisson": lam_m03, "exponential": beta_l01, "normal": mu_m03})
varianz = pd.Series({"binomial": 50 * p_l03 * (1 - p_l03), "poisson": lam_m03, "exponential": beta_l01 ** 2,
                     "normal": sigma_m03 ** 2})
print(pd.DataFrame({"mittel / E": ziehungen.mean() / erwartung, "var / Var": ziehungen.var() / varianz}).round(3))

## 2 · Simulieren statt rechnen: Monte Carlo
*Lerneinheit 2 · Skript Abschnitt 2*

Mit $R$ simulierten Läufen schätzt du eine Wahrscheinlichkeit als Anteil der Läufe, in denen das Ereignis eintritt:
$$\hat p = \frac{1}{R} \sum_{r=1}^{R} \mathbb{1}[\text{Ereignis in Lauf } r], \qquad \mathrm{SE}(\hat p) = \sqrt{\frac{\hat p(1-\hat p)}{R}}.$$
Für eine gewünschte Genauigkeit brauchst du $R = p(1-p)/\mathrm{SE}^2$ Läufe. Erst eine Simulation, deren Antwort du kennst: $P(X = 0)$ für M-03.

In [ ]:
rng = np.random.default_rng(1)
R = 100_000
x_pois = rng.poisson(lam_m03, size=R)
p_dach = (x_pois == 0).mean()
se_dach = math.sqrt(p_dach * (1 - p_dach) / R)
print(round(math.exp(-lam_m03), 4), round(p_dach, 4), round(se_dach, 4))

Dann eine Frage ohne einfache Formel: der **Gesamtstillstand einer Schicht an Linie 1**. Das Modell aus dem Skript: Die Zahl der Störungen an den fünf Maschinen ist Poisson-verteilt mit dem Mittelwert aus allen Schichten, jede Dauer wird mit Zurücklegen aus den beobachteten Dauern der Linie gezogen, der Gesamtstillstand ist die Summe. Simuliert wird ohne Python-Schleife: Für jeden Lauf ziehst du `k.max()` Dauern auf Vorrat, und eine Maske zählt nur die ersten $k_r$.

In [ ]:
linie1 = maschinen.loc[maschinen["linie"] == 1, "maschine"].tolist()
st_l1 = stoerungen[stoerungen["maschine"].isin(linie1)]
anzahl_l1 = st_l1.groupby(["datum", "schicht"], observed=True).size().reindex(alle_schichten, fill_value=0)
summe_l1 = st_l1.groupby(["datum", "schicht"], observed=True)["stillstand_min"].sum().reindex(alle_schichten, fill_value=0)
lam_l1 = anzahl_l1.mean()
dauern_l1 = st_l1["stillstand_min"].to_numpy()
print(linie1, len(dauern_l1), round(lam_l1, 3))
print(round((summe_l1 > 120).mean(), 4), round((summe_l1 > 240).mean(), 4))

rng = np.random.default_rng(5)
k = rng.poisson(lam_l1, size=R)
ziehung = rng.choice(dauern_l1, size=(R, k.max()))
gesamt = (ziehung * (np.arange(k.max()) < k[:, None])).sum(axis=1)
print(k[:5], gesamt[:5], k.max())
for grenze in [120, 240]:
    p_grenze = (gesamt > grenze).mean()
    print(grenze, round(p_grenze, 4), round(math.sqrt(p_grenze * (1 - p_grenze) / R), 4))

### TODO 6 · Monte Carlo für den Gesamtstillstand an Linie 2
Die Gehäusefertigung, Linie 2, hat ebenfalls fünf Maschinen.
1. `linie2`: die Liste ihrer Maschinen. `lam_l2`: die mittlere Zahl der Störungen an diesen Maschinen je Schicht, **mit** den Schichten ohne Störung. `dauern_l2`: alle beobachteten Stillstandsdauern der Linie 2 als Array. `summe_l2`: der beobachtete Gesamtstillstand je Schicht, wieder mit allen Schichten, und `beob120_l2`: der Anteil der Schichten mit mehr als 120 Minuten.
2. `gesamt_l2`: 100.000 simulierte Schichten nach dem Modell oben. `p120_l2`: der simulierte Anteil mit mehr als 120 Minuten, `se120_l2`: sein Monte-Carlo-Standardfehler.

In [ ]:
# TODO 6a
linie2 = None
lam_l2 = None
dauern_l2 = None
summe_l2 = None
beob120_l2 = None

# TODO 6b
rng = np.random.default_rng(6)
gesamt_l2 = None
p120_l2 = None
se120_l2 = None

print(linie2, lam_l2, beob120_l2)
print(p120_l2, se120_l2)

In [ ]:
check("TODO 6a Zutaten", lambda: list(linie2) == ["M-01", "M-02", "M-11", "M-12", "M-13"] and math.isclose(lam_l2, 2.5190, abs_tol=1e-4)
      and len(dauern_l2) == 3645 and len(summe_l2) == 1447 and math.isclose(beob120_l2, 0.28680, abs_tol=1e-5))
check("TODO 6b Simulation", lambda: len(gesamt_l2) >= 10_000 and gesamt_l2.min() >= 0
      and abs(p120_l2 - 0.2849) <= 4 * math.sqrt(0.2849 * 0.7151 / len(gesamt_l2)) + 0.001)
check("TODO 6c Standardfehler", lambda: math.isclose(se120_l2, math.sqrt(p120_l2 * (1 - p120_l2) / len(gesamt_l2)), rel_tol=1e-6))

Simulation und Beobachtung liegen für Linie 2 nah beieinander: beobachtet 28,7 %, simuliert mit Seed 6 ebenfalls 28,7 %. Der Standardfehler der Simulation beträgt mit 100.000 Läufen etwa 0,14 Prozentpunkte, der beobachtete Anteil aus 1.447 Schichten schwankt viel stärker. Mehr Läufe machen die Simulation genauer, aber nicht das Modell richtiger.

## 3 · Gesetz der großen Zahlen und zentraler Grenzwertsatz
*Lerneinheit 3 · Skript Abschnitt 3*

- **Gesetz der großen Zahlen:** Der Mittelwert $\bar X_n$ unabhängiger, gleich verteilter Werte strebt gegen den Erwartungswert $\mu$, wenn $n$ wächst. Er liegt in etwa 95 % der Fälle im Trichter $\mu \pm 2\sigma/\sqrt{n}$. Voraussetzung ist ein endlicher Erwartungswert.
- **Zentraler Grenzwertsatz:** Für großes $n$ ist $\bar X_n$ näherungsweise normalverteilt mit $\mu$ und Standardabweichung $\sigma/\sqrt{n}$, egal wie die einzelnen Werte verteilt sind. Je schiefer die Ausgangsverteilung, desto größer muss $n$ sein.

Der laufende Anteil verspäteter Lieferungen von L-01 in zufälliger Reihenfolge, wie im Skript. `np.cumsum` summiert auf, geteilt durch $1, 2, \dots, n$ ergibt das den Mittelwert bis hierher:

In [ ]:
spaet_l01 = (verzug_l01 > 2).to_numpy()
rng = np.random.default_rng(3)
zufallsfolge = rng.permutation(spaet_l01)
laufend = np.cumsum(zufallsfolge) / np.arange(1, len(zufallsfolge) + 1)
print(round(laufend[9], 3), round(laufend[99], 3), round(laufend[999], 3), round(laufend[-1], 4))

### TODO 7 · Gesetz der großen Zahlen für den mittleren Verzug
1. `laufend_verzug`: der laufende Mittelwert der 1.179 Verzüge von L-01, in zufälliger Reihenfolge (`rng.permutation`).
2. Zeichne in `fig_lln, ax_lln` den laufenden Mittelwert über der Zahl der Lieferungen, eine waagerechte Linie beim Gesamtmittel und den Trichter $\bar x \pm 2s/\sqrt{n}$ als zwei Linien oder als Fläche (`ax.fill_between`). Die x-Achse wird logarithmisch (`ax_lln.set_xscale("log")`), beide Achsen sind beschriftet.

In [ ]:
# TODO 7a
rng = np.random.default_rng(7)
laufend_verzug = None

# TODO 7b
fig_lln, ax_lln = plt.subplots(figsize=(8, 4.5))

plt.show()
print(None if laufend_verzug is None else laufend_verzug[[9, 99, 999, -1]])

In [ ]:
def ist_permutation(laufend_werte, daten):
    """Prüft, ob ein laufender Mittelwert aus einer Umordnung der Daten entstanden ist."""
    einzel = np.diff(np.concatenate([[0.0], np.asarray(laufend_werte) * np.arange(1, len(daten) + 1)]))
    return np.allclose(np.sort(einzel), np.sort(np.asarray(daten, dtype=float)), atol=1e-6)

check("TODO 7a laufender Mittelwert", lambda: len(laufend_verzug) == 1179 and math.isclose(laufend_verzug[-1], 4.2406, abs_tol=1e-4)
      and ist_permutation(laufend_verzug, verzug_l01)
      and not np.allclose(laufend_verzug, np.cumsum(verzug_l01.to_numpy()) / np.arange(1, 1180)))
check("TODO 7b Grafik mit Trichter", lambda: ax_lln.get_xscale() == "log" and len(ax_lln.lines) >= 2
      and (len(ax_lln.lines) >= 4 or len(ax_lln.collections) >= 1) and ax_lln.get_xlabel() != "" and ax_lln.get_ylabel() != "")

Am Anfang springt der Mittelwert wild, ein einziger Verzug von über 100 Stunden verschiebt ihn um Stunden. Mit wachsendem $n$ beruhigt er sich und bleibt im Trichter. Am Ende steht genau der Gesamtmittelwert, denn dann sind alle Lieferungen gezählt.

Der zentrale Grenzwertsatz an den Verzügen von L-01 wie im Skript: Die 1.179 Werte sind die Grundgesamtheit, gezogen wird mit Zurücklegen. `pop.std()` teilt durch $n$, weil es um die Streuung der Grundgesamtheit geht.

In [ ]:
pop = verzug_l01.to_numpy()
rng = np.random.default_rng(4)
for n in [1, 5, 30]:
    mittel = pop[rng.integers(0, len(pop), size=(10_000, n))].mean(axis=1)
    print(n, round(mittel.std(), 2), round(pop.std() / math.sqrt(n), 2), round(pd.Series(mittel).skew(), 2))

### TODO 8 · Zentraler Grenzwertsatz bei L-03
1. `pop_l03`: die 809 Verzüge von L-03 als Array. `mittel_n`: ein Dictionary mit den Schlüsseln 1, 5 und 30, darin je 10.000 Mittelwerte aus $n$ Verzügen, mit Zurücklegen gezogen.
2. Zeichne in `fig_zgs, axes_zgs` drei Histogramme nebeneinander, eines je $n$, mit beschrifteten x-Achsen.

Der Selbsttest prüft, ob die Standardabweichung der Mittelwerte zu $\sigma/\sqrt{n}$ passt.

In [ ]:
# TODO 8a
rng = np.random.default_rng(8)
pop_l03 = None
mittel_n = {}

# TODO 8b
fig_zgs, axes_zgs = plt.subplots(1, 3, figsize=(12, 3.5))

plt.show()

In [ ]:
SIGMA_L03, MU_L03 = 21.8937, 10.8614
check("TODO 8a Mittelwerte", lambda: set(mittel_n) == {1, 5, 30} and len(pop_l03) == 809
      and all(len(mittel_n[n]) == 10_000 for n in [1, 5, 30])
      and all(abs(np.std(mittel_n[n]) / (SIGMA_L03 / math.sqrt(n)) - 1) < 0.1 for n in [1, 5, 30])
      and all(abs(np.mean(mittel_n[n]) - MU_L03) < 4 * SIGMA_L03 / math.sqrt(10_000 * n) for n in [1, 5, 30]))
check("TODO 8b drei Histogramme", lambda: len(axes_zgs) == 3 and all(len(a.patches) >= 10 and a.get_xlabel() != "" for a in axes_zgs))

In [ ]:
if not mittel_n:
    print("⏳ Erst TODO 8 lösen.")
else:
    print(pd.DataFrame({"Streuung": {n: np.std(v) for n, v in mittel_n.items()},
                        "sigma / Wurzel n": {n: SIGMA_L03 / math.sqrt(n) for n in mittel_n},
                        "Schiefe": {n: pd.Series(v).skew() for n, v in mittel_n.items()}}).round(2))

Die Streuung schrumpft wie $\sigma/\sqrt{n}$, die Schiefe ungefähr wie $1/\sqrt{n}$. Die Verzüge von L-03 sind weniger schief als die von L-01, ihre Mittelwerte aus 30 sind entsprechend etwas glockenförmiger.

Auch die Binomialverteilung ist eine Summe und für großes $n$ näherungsweise normal. Mit der Grenze 34,5 statt 35, weil die Normalverteilung stetig ist:

In [ ]:
p_spaet = spaet_l01.mean()
exakt = sum(math.comb(100, j) * p_spaet ** j * (1 - p_spaet) ** (100 - j) for j in range(35, 101))
genaehert = 1 - Phi((34.5 - 100 * p_spaet) / math.sqrt(100 * p_spaet * (1 - p_spaet)))
print(round(p_spaet, 4), round(exakt, 4), round(genaehert, 4))

## 4 · Der Standardfehler
*Lerneinheit 4 · Skript Abschnitt 4*

Die **Standardabweichung** $s$ beschreibt die Streuung der einzelnen Werte. Der **Standardfehler** beschreibt die Streuung eines Schätzers über gedachte Wiederholungen der Erhebung:
$$\mathrm{SE}(\bar x) = \frac{s}{\sqrt{n}}, \qquad \mathrm{SE}(\hat p) = \sqrt{\frac{\hat p(1-\hat p)}{n}}, \qquad \text{95-\%-Intervall: } \bar x \pm 1{,}96\,\mathrm{SE}.$$
Viermal so viele Daten halbieren den Standardfehler. pandas rechnet ihn mit `x.sem()`.

In [ ]:
verzug_l05 = lieferungen.loc[lieferungen["lieferant"] == "L-05", "verzug_h"]
for name, x in [("L-01", verzug_l01), ("L-05", verzug_l05)]:
    print(name, len(x), round(x.mean(), 2), round(x.std(), 2), round(x.std() / math.sqrt(len(x)), 3), round(x.sem(), 3))

fig, ax = plt.subplots(figsize=(8, 3))
mittel_zwei = [verzug_l01.mean(), verzug_l05.mean()]
halbe_breite = [1.96 * verzug_l01.sem(), 1.96 * verzug_l05.sem()]
ax.errorbar(mittel_zwei, [0, 1], xerr=halbe_breite, fmt="o", capsize=5)
ax.set_yticks([0, 1], ["L-01", "L-05"])
ax.set_xlabel("mittlerer Verzug in Stunden, Balken: 95-%-Intervall")
plt.show()

### TODO 9 · Mittlerer Verzug je Lieferant mit Standardfehler
1. `tabelle_se`: ein DataFrame mit einer Zeile je Lieferant (Index `L-01` bis `L-06`) und den Spalten `n`, `mittel`, `std`, `se`, `unten` und `oben`, den Grenzen des 95-%-Intervalls $\bar x \pm 1{,}96\,\mathrm{SE}$. Benannte Aggregation aus Modul 2 hilft beim Anfang: `.agg(n="size", mittel="mean", std="std")`.
2. Zeichne in `fig_fb, ax_fb` die sechs Mittelwerte mit ihren 95-%-Intervallen als Fehlerbalken (`ax_fb.errorbar`), mit einer Achsenbeschriftung, die sagt, was die Balken zeigen.

In [ ]:
# TODO 9a
tabelle_se = None

# TODO 9b
fig_fb, ax_fb = plt.subplots(figsize=(8, 4.5))

plt.show()
print(None if tabelle_se is None else tabelle_se.round(3))

In [ ]:
def zeigt_mittelwerte(ax, werte):
    werte = np.sort(np.asarray(werte, dtype=float))
    return any(len(l.get_xdata()) == len(werte) and (np.allclose(np.sort(np.asarray(l.get_xdata(), dtype=float)), werte)
               or np.allclose(np.sort(np.asarray(l.get_ydata(), dtype=float)), werte)) for l in ax.lines)

check("TODO 9a Form", lambda: list(tabelle_se.index) == ["L-01", "L-02", "L-03", "L-04", "L-05", "L-06"]
      and {"n", "mittel", "std", "se", "unten", "oben"} <= set(tabelle_se.columns) and [int(v) for v in tabelle_se["n"]] == [1179, 967, 809, 1011, 30, 628])
check("TODO 9b Standardfehler", lambda: math.isclose(tabelle_se.loc["L-05", "se"], 2.7755, abs_tol=1e-4)
      and math.isclose(tabelle_se.loc["L-01", "se"], 0.37098, abs_tol=1e-5) and math.isclose(tabelle_se.loc["L-04", "se"], 0.15425, abs_tol=1e-5))
check("TODO 9c Intervalle", lambda: np.allclose(tabelle_se["unten"], tabelle_se["mittel"] - 1.96 * tabelle_se["se"])
      and np.allclose(tabelle_se["oben"], tabelle_se["mittel"] + 1.96 * tabelle_se["se"]) and math.isclose(tabelle_se.loc["L-05", "unten"], 1.8201, abs_tol=1e-4))
check("TODO 9d Fehlerbalken", lambda: any(type(c).__name__ == "ErrorbarContainer" for c in ax_fb.containers)
      and zeigt_mittelwerte(ax_fb, tabelle_se["mittel"]) and (ax_fb.get_xlabel() != "" or ax_fb.get_ylabel() != ""))

Der Standardfehler von L-05 ist 7,5-mal so groß wie der von L-01, das 95-%-Intervall reicht von 1,8 bis 12,7 Stunden. L-03 kommt am spätesten und klar später als alle Lieferanten mit vielen Lieferungen, L-04 fast pünktlich. Ob L-05 unzuverlässiger ist als L-01, klärt der Permutationstest in Abschnitt 6.

Trifft das Intervall $\bar x \pm 1{,}96\,\mathrm{SE}$ wirklich in 95 % der Fälle? Mit den 1.179 Verzügen von L-01 als Grundgesamtheit und Stichproben von $n = 30$, wie im Skript:

In [ ]:
rng = np.random.default_rng(6)
mu_pop = pop.mean()
stichproben = pop[rng.integers(0, len(pop), size=(10_000, 30))]
mittel30 = stichproben.mean(axis=1)
se30 = stichproben.std(axis=1, ddof=1) / math.sqrt(30)
trifft = (mittel30 - 1.96 * se30 <= mu_pop) & (mu_pop <= mittel30 + 1.96 * se30)
print(round(mu_pop, 4), round(trifft.mean(), 4))
print(round((mittel30 + 1.96 * se30 < mu_pop).mean(), 4), round((mittel30 - 1.96 * se30 > mu_pop).mean(), 4))

### TODO 10 · Überdeckung für verschiedene n
`ueberdeckung`: eine Series mit dem Index 5, 30, 100 und 300. Darin steht für jedes $n$ der Anteil von 10.000 Intervallen $\bar x \pm 1{,}96\,\mathrm{SE}$ aus Stichproben der Größe $n$ (mit Zurücklegen aus `pop`), die den Mittelwert `mu_pop` treffen. Rechne $s$ mit `ddof=1`.

In [ ]:
# TODO 10
rng = np.random.default_rng(10)
ueberdeckung = None

print(ueberdeckung)

In [ ]:
UEBERDECKUNG_REF = {5: 0.6803, 30: 0.8823, 100: 0.9260, 300: 0.9419}
check("TODO 10 Überdeckung", lambda: list(ueberdeckung.index) == [5, 30, 100, 300]
      and all(abs(ueberdeckung[n] - c) <= 4 * math.sqrt(c * (1 - c) / 10_000) + 0.002 for n, c in UEBERDECKUNG_REF.items())
      and ueberdeckung.is_monotonic_increasing)

Mit fünf Werten trifft das Intervall nur in gut zwei Dritteln der Fälle, mit 300 fast in 95 %. Bei schiefen Daten fehlen in kleinen Stichproben oft die seltenen langen Verzüge, dann sind Mittelwert und Standardabweichung zugleich zu klein.

## 5 · Bootstrap
*Lerneinheit 5 · Skript Abschnitt 5*

Der **Bootstrap** (Efron 1979) ersetzt die unbekannte Grundgesamtheit durch die Stichprobe:
1. Ziehe aus den $n$ Werten $n$ **mit Zurücklegen**.
2. Berechne die Statistik für diese Bootstrap-Stichprobe.
3. Wiederhole das $B$-mal.

Die Standardabweichung der $B$ Werte ist der **Bootstrap-Standardfehler**, ihr 2,5-%- und 97,5-%-Quantil bilden das **Perzentil-Intervall**. Erst an fünf Werten, dann als Prüfstein am Mittelwert von L-01, wo die Formel $s/\sqrt{n}$ bekannt ist:

In [ ]:
fuenf = np.array([-2.0, -1.0, 0.0, 3.0, 15.0])
rng = np.random.default_rng(7)
boot_fuenf = np.array([rng.choice(fuenf, size=5).mean() for _ in range(5000)])
print(round(boot_fuenf.std(ddof=1), 3), round(fuenf.std(ddof=1) / math.sqrt(5), 3), round(fuenf.std() / math.sqrt(5), 3))
print(np.percentile(boot_fuenf, [2.5, 97.5]))

rng = np.random.default_rng(8)
werte_l01 = verzug_l01.to_numpy()
boot_mittel = np.array([rng.choice(werte_l01, size=len(werte_l01)).mean() for _ in range(5000)])
print(round(boot_mittel.std(ddof=1), 3), round(verzug_l01.sem(), 3))
print(np.percentile(boot_mittel, [2.5, 97.5]).round(2))

### TODO 11 · Eigene Bootstrap-Funktion für den Median
1. `bootstrap(x, statistik, B, rng)`: gibt ein Array mit `B` Bootstrap-Werten der Funktion `statistik` zurück. Jede Bootstrap-Stichprobe zieht `len(x)` Werte aus `x` mit Zurücklegen, mit dem übergebenen Generator `rng`. `statistik` ist eine Funktion wie `np.median` oder `np.mean`.
2. `boot_med_l05` und `boot_med_l01`: je 5.000 Bootstrap-Mediane des Verzugs von L-05 und L-01. `ki_l05` und `ki_l01`: die 95-%-Perzentil-Intervalle als Arrays mit zwei Werten (`np.percentile(..., [2.5, 97.5])`).

In [ ]:
# TODO 11a
def bootstrap(x, statistik, B, rng):
    return None            # ersetze diese Zeile

# TODO 11b
rng = np.random.default_rng(11)
boot_med_l05 = None
boot_med_l01 = None
ki_l05 = None
ki_l01 = None

print(ki_l05, ki_l01)

In [ ]:
probe_boot = None
try:
    probe_boot = bootstrap(np.array([1.0, 2.0, 3.0, 4.0]), np.mean, 2000, np.random.default_rng(0))
except Exception:
    pass
check("TODO 11a bootstrap", lambda: len(probe_boot) == 2000 and probe_boot.min() >= 1 and probe_boot.max() <= 4
      and abs(np.std(probe_boot) / math.sqrt(1.25 / 4) - 1) < 0.1)
check("TODO 11b Bootstrap-Mediane", lambda: len(boot_med_l05) == 5000 and len(boot_med_l01) == 5000
      and -1.2 <= boot_med_l01.min() and boot_med_l01.max() <= -0.2 and len(np.unique(boot_med_l01)) <= 15
      and len(np.unique(boot_med_l05)) >= 20)
check("TODO 11c Perzentil-Intervalle", lambda: len(ki_l05) == 2 and -2.6 <= ki_l05[0] <= -1.2 and 4.0 <= ki_l05[1] <= 7.5
      and -0.9 <= ki_l01[0] <= -0.7 and -0.6 <= ki_l01[1] <= -0.4)

In [ ]:
if boot_med_l05 is None or boot_med_l01 is None:
    print("⏳ Erst TODO 11 lösen.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    for ax_b, werte_b, name_b in [(axes[0], boot_med_l05, "L-05"), (axes[1], boot_med_l01, "L-01")]:
        stufen, haeufigkeit = np.unique(np.round(werte_b, 2), return_counts=True)
        ax_b.bar(stufen, haeufigkeit / len(werte_b), width=0.04)
        ax_b.set_title(f"{name_b}: Bootstrap-Mediane, {len(stufen)} verschiedene Werte")
    axes[1].set_xlabel("Median der Bootstrap-Stichprobe in Stunden")
    plt.show()

Der Median von L-05 ist unsicher, das Intervall reicht von etwa −1,9 bis 5,6 Stunden. Seine Bootstrap-Verteilung ist zackig, weil der Median aus 30 Werten nur wenige verschiedene Werte annehmen kann. Der Median von L-01 liegt eng zwischen −0,8 und −0,5 Stunden.

## 6 · Permutationstest
*Lerneinheit 6 · Skript Abschnitt 6*

Ist L-05 unzuverlässiger als L-01? Der **Permutationstest** (Fisher 1935, Pitman 1937) prüft die Nullhypothese $H_0$: Die Zuordnung zum Lieferanten spielt keine Rolle, die Etiketten sind vertauschbar.
1. Berechne die Teststatistik $T$ für die echten Etiketten, etwa die Differenz der Mittelwerte.
2. Mische die Etiketten $R$-mal (`rng.permutation`) und berechne $T$ jedes Mal neu.
3. Zähle, wie viele Mischungen mindestens so extrem sind wie die Beobachtung, in **beide** Richtungen: $b = \#\{|T^*| \ge |T|\}$.
4. Der p-Wert ist $p = (b + 1)/(R + 1)$ (Phipson und Smyth 2010).

Das Mini-Beispiel aus dem Skript mit allen sechs Aufteilungen und der Test für den Anteil verspäteter Lieferungen:

In [ ]:
from itertools import combinations

mini = np.array([6.0, 10.0, 1.0, 3.0])
mini_diff = []
for auswahl in combinations(range(4), 2):
    gruppe_a = mini[list(auswahl)]
    gruppe_b = np.delete(mini, list(auswahl))
    mini_diff.append(float(gruppe_a.mean() - gruppe_b.mean()))
print(mini_diff, round(np.mean(np.abs(mini_diff) >= 6), 4))

spaet_l05 = (verzug_l05 > 2).to_numpy()
beob_spaet = spaet_l05.mean() - spaet_l01.mean()
zusammen = np.concatenate([spaet_l05, spaet_l01])
rng = np.random.default_rng(10)
perm_spaet = np.empty(10_000)
for r in range(10_000):
    gemischt = rng.permutation(zusammen)
    perm_spaet[r] = gemischt[:30].mean() - gemischt[30:].mean()
b_spaet = (np.abs(perm_spaet) >= abs(beob_spaet)).sum()
print(round(beob_spaet, 4), b_spaet, round((b_spaet + 1) / (10_000 + 1), 4))

### TODO 12 · Eigener Permutationstest für den mittleren Verzug
1. `perm_test(a, b, R, rng)`: ein zweiseitiger Permutationstest für die Differenz der Mittelwerte `mean(a) - mean(b)`. Die Funktion gibt drei Dinge zurück: die beobachtete Differenz, den p-Wert mit der Korrektur $+1$ aus Schritt 4 und das Array der `R` permutierten Differenzen. Nenne die Zahl der extremen Mischungen nicht `b`, der Name gehört schon dem zweiten Argument.
2. `ergebnis_l05`: das Ergebnis für L-05 gegen L-01, `ergebnis_l03`: für L-03 gegen L-01, je mit `R = 10_000`.
3. Zeichne in `fig_perm, axes_perm` zwei Histogramme der Permutationsverteilungen, links L-05, rechts L-03, mit der beobachteten Differenz als senkrechter Linie (`ax.axvline`).

Der Selbsttest probiert deine Funktion auch am Mini-Beispiel aus.

In [ ]:
# TODO 12a
def perm_test(a, b, R, rng):
    return None, None, None            # ersetze diese Zeile: beobachtet, p, verteilung

# TODO 12b
rng = np.random.default_rng(12)
ergebnis_l05 = None
ergebnis_l03 = None

# TODO 12c
fig_perm, axes_perm = plt.subplots(1, 2, figsize=(11, 4))

plt.show()

In [ ]:
probe_perm = (None, None, None)
try:
    probe_perm = perm_test([6.0, 10.0], [1.0, 3.0], 3000, np.random.default_rng(0))
except Exception:
    pass
check("TODO 12a perm_test am Mini-Beispiel", lambda: probe_perm[0] == 6.0 and len(probe_perm[2]) == 3000
      and abs(probe_perm[1] - 1 / 3) <= 4 * math.sqrt(2 / 9 / 3000) + 1 / 3001)
check("TODO 12b L-05 gegen L-01", lambda: math.isclose(ergebnis_l05[0], 3.0194, abs_tol=1e-4) and len(ergebnis_l05[2]) == 10_000
      and 0.15 <= ergebnis_l05[1] <= 0.24)
check("TODO 12c L-03 gegen L-01", lambda: math.isclose(ergebnis_l03[0], 6.6208, abs_tol=1e-4)
      and 1 / (len(ergebnis_l03[2]) + 1) - 1e-12 <= ergebnis_l03[1] <= 0.001)
check("TODO 12d zwei Histogramme", lambda: len(axes_perm) == 2 and all(len(a.patches) >= 10 and len(a.lines) >= 1 for a in axes_perm))

L-05 liegt im Mittel 3,02 Stunden über L-01, aber in fast jeder fünften Mischung ist der Unterschied mindestens so groß: $p \approx 0{,}2$, **kein Nachweis**. Das heißt nicht, dass beide gleich zuverlässig sind. Mit 30 Lieferungen ließe sich auch ein echter Unterschied von drei Stunden schwer nachweisen, die Teststärke kommt in Modul 6. Bei L-03 erreicht keine Mischung die beobachtete Differenz, $p = 1/10001 < 0{,}001$.

## 7 · Seeds und Reproduzierbarkeit
*Lerneinheit 7 · Skript Abschnitt 7*

Pseudozufallszahlen sind aus dem Seed berechnet: gleicher Seed, gleiche Folge. `np.random.default_rng(seed)` legt einen eigenen Generator an (PCG64), den du als Argument weitergibst. Das alte `np.random.seed` setzt einen globalen Zustand, den jeder zusätzliche Aufruf verschiebt. `SeedSequence(seed).spawn(k)` erzeugt unabhängige Ströme für parallele Simulationen.

In [ ]:
a = np.random.default_rng(2026).integers(0, 100, size=5)
b = np.random.default_rng(2026).integers(0, 100, size=5)
c = np.random.default_rng(2027).integers(0, 100, size=5)
print(a, b, c, np.array_equal(a, b))
print(type(np.random.default_rng(2026).bit_generator).__name__)

np.random.seed(1)
vorher = np.random.poisson(lam_l1, size=5)
np.random.seed(1)
np.random.random()
nachher = np.random.poisson(lam_l1, size=5)
print(vorher, nachher)
kinder = np.random.SeedSequence(2026).spawn(3)
stroeme = [np.random.default_rng(kind) for kind in kinder]
print([int(g.integers(0, 100)) for g in stroeme])

### TODO 13 · Gleicher Seed, anderer Seed, mehr Läufe
Du brauchst `perm_test` aus TODO 12.
1. `p_seed13_a` und `p_seed13_b`: der p-Wert für L-05 gegen L-01 mit `R = 1000`, beide Male mit einem frischen Generator mit Seed 13. `p_seed14`: dasselbe mit Seed 14.
2. `p_seeds`: ein Dictionary mit den Schlüsseln 200 und 2000. Darin je ein Array mit den p-Werten für die Seeds 0 bis 9, einmal mit `R = 200` und einmal mit `R = 2000`. Nimm für jeden Seed einen frischen Generator, etwa `np.random.default_rng(1300 + s)` für `s` von 0 bis 9.

In [ ]:
# TODO 13a
p_seed13_a = None
p_seed13_b = None
p_seed14 = None

# TODO 13b
p_seeds = {200: None, 2000: None}

print(p_seed13_a, p_seed13_b, p_seed14)
print(p_seeds)

In [ ]:
check("TODO 13a gleicher Seed, gleiches Ergebnis", lambda: p_seed13_a == p_seed13_b and 0.12 <= p_seed13_a <= 0.27 and 0.12 <= p_seed14 <= 0.27)
check("TODO 13b p-Werte über Seeds", lambda: set(p_seeds) == {200, 2000} and all(len(p_seeds[r]) == 10 for r in p_seeds)
      and all(0.08 <= v <= 0.32 for v in p_seeds[200]) and all(0.15 <= v <= 0.24 for v in p_seeds[2000])
      and np.std(p_seeds[200]) > np.std(p_seeds[2000]))

In [ ]:
if not all(isinstance(p_seeds.get(r_s), np.ndarray) and p_seeds[r_s].dtype.kind == "f" for r_s in [200, 2000]):
    print("⏳ Erst TODO 12 und TODO 13 lösen.")
else:
    for R_zeige in [200, 2000]:
        werte_s = p_seeds[R_zeige]
        print(f"R = {R_zeige}: p von {werte_s.min():.3f} bis {werte_s.max():.3f}, Streuung {werte_s.std(ddof=1):.4f},",
              f"erwartet etwa {math.sqrt(0.194 * 0.806 / R_zeige):.4f}")

Mit 200 Mischungen schwankt der p-Wert von Seed zu Seed um mehrere Hundertstel, mit 2.000 nur noch um knapp eines. Die Schwankung passt zum Monte-Carlo-Standardfehler $\sqrt{p(1-p)/R}$. Den Seed zu wechseln, bis ein p-Wert passt, ist Manipulation. Erhöhe stattdessen $R$ und nenne Seed, $R$ und Softwareversionen im Bericht.

## Bonus

### Bonus B1 · Mangelquote L-03 gegen L-01
Ist die Mangelquote von L-03 höher als die von L-01? Gemischt werden ganze Lieferungen, jede mit ihren geprüften und mangelhaften Teilen.
1. `perm_quote(a, b, R, rng)`: `a` und `b` sind DataFrames mit den Spalten `mangel_stueck` und `geprueft_stueck`. Die Statistik ist die Differenz der Mangelquoten als Quote der Summen, `a` minus `b`. Die Funktion gibt die beobachtete Differenz und den zweiseitigen p-Wert mit der Korrektur $+1$ zurück.
2. `quote_l03` und `quote_l05`: die Ergebnisse für L-03 gegen L-01 und L-05 gegen L-01 mit je `R = 5000`.

In [ ]:
# Bonus B1
def perm_quote(a, b, R, rng):
    return None, None            # ersetze diese Zeile: beobachtet, p

rng = np.random.default_rng(101)
quote_l03 = None
quote_l05 = None

for name_q, erg_q in [("L-03", quote_l03), ("L-05", quote_l05)]:
    print(name_q, erg_q if erg_q is None else (round(float(erg_q[0]), 5), round(float(erg_q[1]), 4)))

In [ ]:
check("B1 L-03 gegen L-01", lambda: math.isclose(quote_l03[0], 0.0047485, abs_tol=1e-6) and 0 < quote_l03[1] <= 0.001)
check("B1 L-05 gegen L-01", lambda: math.isclose(quote_l05[0], -0.0032869, abs_tol=1e-6) and 0.11 <= quote_l05[1] <= 0.24)

L-03 hat eine um 0,47 Prozentpunkte höhere Mangelquote, keine Mischung kommt da heran. L-05 liegt 0,33 Prozentpunkte unter L-01, aber mit $p \approx 0{,}18$ ist auch das kein Nachweis.

### Bonus B2 · Alle Lieferanten gegen L-01
`p_alle`: eine Series mit den Lieferanten `L-02` bis `L-06` als Index und dem p-Wert von `perm_test` für den mittleren Verzug gegen L-01, je mit `R = 2000` und dem Generator aus der Zelle. Welche Lieferanten haben einen p-Wert unter 0,05? Was würde sich ändern, wenn du alle 15 Paare vergleichst?

In [ ]:
# Bonus B2
rng = np.random.default_rng(102)
p_alle = None

print(p_alle)

In [ ]:
check("B2 p_alle", lambda: list(p_alle.index) == ["L-02", "L-03", "L-04", "L-05", "L-06"]
      and p_alle["L-03"] <= 0.001 and p_alle["L-04"] <= 0.001 and 0.24 <= p_alle["L-02"] <= 0.39
      and 0.15 <= p_alle["L-05"] <= 0.24 and 0.35 <= p_alle["L-06"] <= 0.50)

L-03 kommt später, L-04 früher als L-01, beide mit $p < 0{,}001$. Für L-02, L-05 und L-06 weisen die Daten keinen Unterschied nach. Bei fünf Tests mit der Schwelle 0,05 liegt die Chance auf mindestens einen falschen Treffer schon bei bis zu $1 - 0{,}95^5 \approx 23\,\%$, wenn alle gleich wären, bei allen 15 Paaren bei bis zu 54 %. Wie man damit umgeht, zeigt Modul 6.

## Zum Nachdenken

1. In TODO 3 hast du die Schichten ohne Störung aus `produktion.csv` ergänzt. Warum wäre es falsch, stattdessen für jeden Kalendertag drei Schichten mit 0 Störungen anzunehmen?
2. Die Simulation für Linie 2 trifft den Anteil über 120 Minuten gut. Welche Annahme des Modells würdest du als Erstes prüfen, wenn sie für über 240 Minuten deutlich danebenläge?
3. In TODO 10 steigt die Trefferquote mit $n$. Wie sähe sie bei den Wellendurchmessern von M-09 aus, die fast symmetrisch normalverteilt sind?
4. Die obere Grenze des Bootstrap-Intervalls für den Median von L-05 liegt bei den meisten Seeds bei 5,55 Stunden, bei einigen bei 6,1 Stunden. Wie würdest du das Intervall im Bericht angeben?
5. Der Einkauf liest $p \approx 0{,}2$ und schreibt: „L-05 ist genauso zuverlässig wie L-01.“ Was antwortest du?

<details><summary>Hinweise zu den Antworten</summary>

1. An Sonntagen, Feiertagen und in den Werksferien wird nicht gearbeitet, die Maschine steht, und eine Störung ist gar nicht möglich. Solche Tage würden den Mittelwert drücken und die Verteilung verzerren. Gezählt werden nur Schichten, in denen gearbeitet wurde.
2. Die Unabhängigkeit: Hängen lange Störungen zusammen, etwa weil eine Maschine ausfällt und die nächste mit, gibt es mehr sehr lange Gesamtstillstände als im Modell. Auch eine Rate, die zwischen Schichten oder Jahreszeiten schwankt, macht die Ränder breiter.
3. Deutlich näher an 95 %: Bei fast symmetrischen Daten ohne lange Ränder ist der Mittelwert schnell normalverteilt, und $s$ ist nicht systematisch zu klein. Bei sehr kleinem $n$ bleibt die Quote trotzdem etwas unter 95 %, weil auch $s$ schwankt, die Korrektur dafür kommt in Modul 6. Probiere es mit den Messwerten aus.
4. Mit dem Seed, der Zahl der Bootstrap-Stichproben und einer sinnvollen Rundung, etwa „−1,9 bis knapp 6 Stunden (B = 5.000)“. Die zweite Nachkommastelle täuscht eine Genauigkeit vor, die es nicht gibt.
5. Kein Nachweis eines Unterschieds ist kein Nachweis der Gleichheit. Mit 30 Lieferungen ist der Standardfehler fast drei Stunden groß, auch ein echter Unterschied von drei Stunden bliebe oft unentdeckt. Besser: weiter Daten sammeln und vorher festlegen, welcher Unterschied für den Einkauf zählt.
</details>

## Wie es weitergeht

Du hast den Zufall simuliert, statt Formeln zu rechnen: Monte Carlo für Wahrscheinlichkeiten, den Bootstrap für die Unsicherheit eines Medians und den Permutationstest für die Frage „echt oder Zufall?“. **Modul 6 „Schätzen und Testen“** zeigt dieselben Fragen mit Formeln: Konfidenzintervalle und Tests, die Teststärke, mit der du planst, wie viele Daten du brauchst, und den Umgang mit vielen Tests auf einmal.